In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

In [11]:
import pandas as pd
from pathlib import Path


In [16]:
DATA_DIR = Path(r"C:\ai\source\09_project\data")
files = {
    "공공자전거_대여소별_이용정보_2020_01~05.csv": "utf-8-sig",
    "공공자전거 대여소별 이용정보_202006.csv": "cp949",
    "공공자전거_대여소별_이용정보_2020년_7월~12월.csv": "utf-8-sig",
}

In [17]:
rename_map = {"대여소 그룹": "자치구", "일자 / 월": "대여년월",
              "대여 일자 / 월": "대여년월", "대여 건수": "대여건수"}
use_cols = ["자치구", "대여소 명", "대여년월", "대여건수"]

frames = []
for fname, enc in files.items():
    df = pd.read_csv(DATA_DIR/fname, encoding=enc)
    df.columns = df.columns.str.strip()     
    df = df.rename(columns=rename_map)
    df = df[use_cols]                        
    print(f"{fname}: {df.shape}")
    frames.append(df)


공공자전거_대여소별_이용정보_2020_01~05.csv: (8459, 4)
공공자전거 대여소별 이용정보_202006.csv: (2062, 4)
공공자전거_대여소별_이용정보_2020년_7월~12월.csv: (12564, 4)


In [18]:
bike = pd.concat(frames,ignore_index=True)
print("병합:",bike.shape)

병합: (23085, 4)


In [19]:
bike["자치구"] = bike["자치구"].astype(str).str.strip()
bike["대여소 명"] = bike["대여소 명"].astype(str).str.strip()
bike["대여년월"] = bike["대여년월"].astype(str).str.strip()     # 예: '202001'
bike["대여건수"] = pd.to_numeric(
    bike["대여건수"].astype(str).str.replace(",", ""), errors="coerce"
)


In [20]:
before = len(bike)
bike = bike[~bike["자치구"].isin(["그룹명 없음", "정비센터"])]
bike = bike[~bike["대여소 명"].isin(["", "대여소명 없음", "nan"])]
bike = bike.dropna(subset=["대여건수"])
print(f"제거된 행: {before - len(bike)}")


제거된 행: 126


In [21]:
dup = bike.duplicated(subset=["자치구", "대여소 명", "대여년월"]).sum()
print("중복 행 수:", dup)
if dup:
    bike = bike.groupby(["자치구", "대여소 명", "대여년월"], as_index=False)["대여건수"].sum()


중복 행 수: 0


In [22]:
bike["대여건수"] = bike["대여건수"].astype(int)
bike = bike.sort_values(["대여년월", "자치구", "대여소 명"]).reset_index(drop=True)

print(bike.info())
print(bike.head())
print(bike["대여년월"].value_counts().sort_index())
print("자치구 수:", bike["자치구"].nunique())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22959 entries, 0 to 22958
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   자치구     22959 non-null  object
 1   대여소 명   22959 non-null  object
 2   대여년월    22959 non-null  object
 3   대여건수    22959 non-null  int32 
dtypes: int32(1), object(3)
memory usage: 627.9+ KB
None
   자치구                           대여소 명    대여년월  대여건수
0  강남구                2301. 현대고등학교 건너편  202001   577
1  강남구  2302. 교보타워 버스정류장(신논현역 3번출구 후면)  202001   690
2  강남구                  2303. 논현역 7번출구  202001   297
3  강남구         2304. 신영 ROYAL PALACE 앞  202001   136
4  강남구              2305. MCM 본사 직영점 앞  202001   212
202001    1534
202002    1541
202003    1721
202004    1869
202005    1748
202006    2051
202007    2067
202008    2069
202009    2079
202010    2075
202011    2076
202012    2129
Name: 대여년월, dtype: int64
자치구 수: 25


In [24]:
bike.to_csv("공공자전거_대여소별_이용정보_2020_전처리.csv", index=False, encoding="utf-8-sig")
